# Huggingface Audio Spectrogram Transformer (potentially this is trained for speech recognition. need to check if adaptable for our purposes)

## Figuring out what I need to import and in what form the data needs to be

from the website: https://huggingface.co/docs/transformers/v4.48.0/en/model_doc/audio-spectrogram-transformer

Usage tips

    When fine-tuning the Audio Spectrogram Transformer (AST) on your own dataset, it’s recommended to take care of the input normalization (to make sure the input has mean of 0 and std of 0.5). ASTFeatureExtractor takes care of this. Note that it uses the AudioSet mean and std by default. You can check ast/src/get_norm_stats.py to see how the authors compute the stats for a downstream dataset.\\
    Note that the AST needs a low learning rate (the authors use a 10 times smaller learning rate compared to their CNN model proposed in the PSLA paper) and converges quickly, so please search for a suitable learning rate and learning rate scheduler for your task.

# TODO how to finetune the ast on the data set 

### load audio file from raw_data and resample as the model wants 16kHz

In [13]:
import soundfile as sf
import librosa as lb
import IPython

#filepath = '/home/sunil/code/mi-mi-mia/smart-stethoscope/preprocessed_data/audio_breathing_cycles/226_1b1_Pl_sc_LittC2SE_9.wav'
filepath = '/home/sunil/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/226_1b1_Pl_sc_LittC2SE.wav'
#waveform, sampling_rate = sf.read(filepath)
waveform, sampling_rate = lb.load(filepath, sr=16000)
print(f"Sampling rate: {sampling_rate} Hz")
print(f"Audio shape: {waveform.shape}")

IPython.display.Audio(waveform, rate=sampling_rate)

Sampling rate: 16000 Hz
Audio shape: (320000,)


## load pretrained autofeature extractor (pretrained on what?)

side note: can we use the astfeatureextractor to normalize etc?

In [19]:
from transformers import AutoFeatureExtractor

model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

## feature extraction

In [21]:
inputs = feature_extractor(waveform, sampling_rate=sampling_rate, return_tensors="pt") #np for numpy array
print(f"Input values shape: {inputs.input_values.shape}")

Input values shape: torch.Size([1, 1024, 128])


## load model (pretrained on speech data, so need to figure out how to train on own data)

In [9]:
from transformers import ASTForAudioClassification
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ASTForAudioClassification.from_pretrained(
    model_id,
    attn_implementation="sdpa",  # Use PyTorch's efficient SDPA for faster inference
).to(device)

Loading weights: 100%|██████████| 203/203 [00:00<00:00, 1588.96it/s]


In [15]:
inputs = inputs.to(device)

with torch.no_grad():
    outputs = model(**inputs)

In [16]:
predicted_class_idx = outputs.logits.argmax(-1).item()
predicted_label = model.config.id2label[predicted_class_idx]
print(f"Predicted class: {predicted_label}")

Predicted class: Breathing


In [17]:
# Show top 5 predictions with probabilities
import torch.nn.functional as F

probs = F.softmax(outputs.logits, dim=-1)
top5_probs, top5_indices = torch.topk(probs, 5)

print("Top 5 predictions:")
for prob, idx in zip(top5_probs[0], top5_indices[0]):
    label = model.config.id2label[idx.item()]
    print(f"  {label}: {prob.item():.2%}")

Top 5 predictions:
  Breathing: 44.06%
  Stomach rumble: 14.89%
  Heart sounds, heartbeat: 8.39%
  Heart murmur: 4.70%
  Snort: 4.41%


# Attempt to train a base ASTClassification model with the raw_data
